# Document Processing with Single-Stage Extraction & Confidence Elicitation

This notebook demonstrates the **Single Stage** verbalized approach, which performs extraction and confidence assessment in a **single LLM call** rather than two separate steps. 

## The 1S-TopK Method

For each field, the LLM is asked to provide its **top-K candidate guesses** (K=4) along with the **probability that each guess is correct** on a continuous scale from 0.0 to 1.0. The top guess (G1) becomes the extracted value, and its probability (P1) becomes the confidence score.

**Pipeline steps:**

1. **OCR Service** - Convert a PDF document to text using AWS Textract
2. **Classification Service** - Classify document pages into sections using Bedrock
3. **Extraction + Assessment (1S-TopK)** - Extract structured information AND confidence scores in a single Bedrock call
4. ~~Assessment Service~~ - **Skipped** (confidence already produced by extraction)

## Why Top-K?

Rather than asking the LLM for a single answer and a confidence score, we ask it to produce **K candidates with probabilities**. This serves two purposes:

1. **Better-calibrated confidence** — By requiring the model to distribute probability mass across alternatives, we obtain verbalized confidence scores.

2. **Reduced overconfidence** — When forced to consider and rank alternatives, the model is less likely to assign near-1.0 confidence to uncertain extractions. The Top-K format makes the model "think about what else it could be" before committing to a probability.


> **Reference**: Tian, K., Mitchell, E., Zhou, A., et al. "Just Ask for Calibration: Strategies for Eliciting Calibrated Confidence Scores from Language Models Fine-Tuned with Human Feedback." EMNLP 2023. [aclanthology.org/2023.emnlp-main.330](https://aclanthology.org/2023.emnlp-main.330/)


## 1. Install Dependencies

The IDP common package supports granular installation through extras. You can install:
- `[core]` - Just core functionality 
- `[ocr]` - OCR service with Textract dependencies
- `[classification]` - Classification service dependencies
- `[extraction]` - Extraction service dependencies
- `[evaluation]` - Evaluation service dependencies
- `[all]` - All of the above

In [1]:
# Let's make sure that modules are autoreloaded
%load_ext autoreload
%autoreload 2

ROOTDIR="../.."
# First uninstall existing package (to ensure we get the latest version)
# %pip uninstall -y idp_common

# # Install the IDP common package with all components in development mode
#%pip install -q -e "{ROOTDIR}/lib/idp_common_pkg[dev, all]"

# # Note: We can also install specific components like:
# # %pip install -q -e "{ROOTDIR}/lib/idp_common_pkg[ocr,classification,extraction,evaluation]"

# # Check installed version
# %pip show idp_common | grep -E "Version|Location"

# # Optionally use a .env file for environment variables
# try:
#     from dotenv import load_dotenv
#     load_dotenv()  
# except ImportError:
#     pass  

## 2. Import Libraries and Set Up Environment

In [2]:
import os
import json
import time
import boto3
import logging
import datetime

# Import base libraries
from idp_common.models import Document, Status, Section, Page
from idp_common import ocr, classification, extraction, assessment, evaluation

# Configure logging 
logging.basicConfig(level=logging.WARNING)  # Set root logger to WARNING (less verbose)
logging.getLogger('idp_common.ocr.service').setLevel(logging.INFO)  # Focus on service logs
logging.getLogger('textractor').setLevel(logging.WARNING)  # Suppress textractor logs
logging.getLogger('idp_common.evaluation.service').setLevel(logging.DEBUG)  # Enable evaluation logs
logging.getLogger('idp_common.assessment.service').setLevel(logging.DEBUG)  # Enable assessment logs
logging.getLogger('idp_common.bedrock.client').setLevel(logging.DEBUG)  # show prompts

# Set environment variables
os.environ['METRIC_NAMESPACE'] = 'IDP-Notebook-1S-TopK-Assessment-Example'
os.environ['AWS_REGION'] = boto3.session.Session().region_name or 'us-east-1'

# Get AWS account ID for unique bucket names
sts_client = boto3.client('sts')
account_id = sts_client.get_caller_identity()["Account"]
region = os.environ['AWS_REGION']

# Define sample PDF path 
SAMPLE_PDF_PATH = f"{ROOTDIR}/samples/fcc-invoices-7429ba503d96718740c2f714598b40a4.pdf"

# Create unique bucket names based on account ID and region
input_bucket_name =  os.getenv("IDP_INPUT_BUCKET_NAME", f"idp-notebook-assess-input-{account_id}-{region}")
output_bucket_name = os.getenv("IDP_OUTPUT_BUCKET_NAME", f"idp-notebook-assess-output-{account_id}-{region}")

print("Environment setup:")
print(f"METRIC_NAMESPACE: {os.environ.get('METRIC_NAMESPACE')}")
print(f"AWS_REGION: {os.environ.get('AWS_REGION')}")
print(f"Input bucket: {input_bucket_name}")
print(f"Output bucket: {output_bucket_name}")
print(f"SAMPLE_PDF_PATH: {SAMPLE_PDF_PATH}")

Environment setup:
METRIC_NAMESPACE: IDP-Notebook-1S-TopK-Assessment-Example
AWS_REGION: us-west-2
Input bucket: idp-notebook-assess-input-195275636621-us-west-2
Output bucket: idp-notebook-assess-output-195275636621-us-west-2
SAMPLE_PDF_PATH: ../../samples/fcc-invoices-7429ba503d96718740c2f714598b40a4.pdf


## 3. Set Up S3 Buckets and Upload Sample File

In [3]:
# Create S3 client
s3_client = boto3.client('s3')

# Helper function to parse S3 URIs
def parse_s3_uri(uri):
    parts = uri.replace("s3://", "").split("/")
    bucket = parts[0]
    key = "/".join(parts[1:])
    return bucket, key

# Helper function to load JSON from S3
def load_json_from_s3(uri):
    bucket, key = parse_s3_uri(uri)
    response = s3_client.get_object(Bucket=bucket, Key=key)
    content = response['Body'].read().decode('utf-8')
    return json.loads(content)

# Function to create a bucket if it doesn't exist
def ensure_bucket_exists(bucket_name):
    try:
        s3_client.head_bucket(Bucket=bucket_name)
        print(f"Bucket {bucket_name} already exists")
    except Exception:
        try:
            if region == 'us-east-1':
                s3_client.create_bucket(Bucket=bucket_name)
            else:
                s3_client.create_bucket(
                    Bucket=bucket_name,
                    CreateBucketConfiguration={'LocationConstraint': region}
                )
            print(f"Created bucket: {bucket_name}")
            
            # Deny any non-TLS request to the bucket, matching the EnforceSSLOnly
            # bucket policy the IDP CloudFormation stack applies to its own buckets.
            partition = boto3.Session().get_partition_for_region(region)
            bucket_arn = f"arn:{partition}:s3:::{bucket_name}"
            s3_client.put_bucket_policy(
                Bucket=bucket_name,
                Policy=json.dumps({
                    "Version": "2012-10-17",
                    "Statement": [{
                        "Sid": "EnforceSSLOnly",
                        "Effect": "Deny",
                        "Principal": "*",
                        "Action": "s3:*",
                        "Resource": [bucket_arn, f"{bucket_arn}/*"],
                        "Condition": {"Bool": {"aws:SecureTransport": "false"}},
                    }],
                }),
            )

            # Wait for bucket to be accessible
            waiter = s3_client.get_waiter('bucket_exists')
            waiter.wait(Bucket=bucket_name)
        except Exception as e:
            print(f"Error creating bucket {bucket_name}: {str(e)}")
            raise

# Ensure both buckets exist
ensure_bucket_exists(input_bucket_name)
ensure_bucket_exists(output_bucket_name)

# Upload the sample file to S3
sample_file_key = "sample-assessment-" + datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S") + ".pdf"
with open(SAMPLE_PDF_PATH, 'rb') as file_data:
    s3_client.upload_fileobj(file_data, input_bucket_name, sample_file_key)

print(f"Uploaded sample file to: s3://{input_bucket_name}/{sample_file_key}")

Bucket idp-notebook-assess-input-195275636621-us-west-2 already exists
Bucket idp-notebook-assess-output-195275636621-us-west-2 already exists
Uploaded sample file to: s3://idp-notebook-assess-input-195275636621-us-west-2/sample-assessment-2026-07-07_21-39-43.pdf


## 4. Set Up Configuration with Assessment

In [4]:
# 1S-TopK Configuration: Combined extraction + confidence assessment in a single step.
# Activated by setting extraction.mode=simple + extraction.confidence.mode=integrated.
# The TopK prompt instructs the LLM to return top-4 guesses with probabilities
# per attribute (G1/P1/.../G4/P4). The ExtractionService resolves this into
# inference_result + explainability_info, so the separate assessment step is skipped.
CONFIG = {
    "ocr": {
        "backend": "textract",
        "features": [
            {
                "name": "LAYOUT"
            }
        ],
        "image": {
            "dpi": None,
            "preprocessing": None,
            "target_height": "800",
            "target_width": "800"
        },
        "max_workers": 20,
        "model_id": "us.amazon.nova-2-lite-v1:0",
        "system_prompt": "You are an expert OCR system. Extract all text from the provided image accurately, preserving layout where possible.",
        "task_prompt": "Extract all text from this document image. Preserve the layout, including paragraphs, tables, and formatting."
    },
    "classification": {
        "classificationMethod": "multimodalPageLevelClassification",
        "contextPagesCount": "0",
        "image": {
            "target_height": "",
            "target_width": ""
        },
        "maxPagesForClassification": "ALL",
        "max_tokens": "4096",
        "model": "us.amazon.nova-pro-v1:0",
        "sectionSplitting": "llm_determined",
        "system_prompt": "You are a multimodal document classification expert that analyzes business documents using both visual layout and textual content. Your task is to classify single-page documents into predefined categories based on their structural patterns, visual features, and text content. Your output must be valid JSON according to the requested format. <variables> <document-ocr-data>: OCR-extracted text content from the document page that provides textual information for classification <document-image>: Visual representation of the document page that provides layout, formatting, and visual structure information <document-types>: List of valid document types with their descriptions that the document must be classified into </variables>",
        "task_prompt": "<task-description> Analyze the provided document using both its visual layout and textual content to determine its document type and whether this page begins a new document or continues the previous one. </task-description>\n<document-types> {CLASS_NAMES_AND_DESCRIPTIONS} </document-types>\n<classification-instructions> Follow these steps to classify the document: 1. Examine the visual layout: headers, logos, formatting, structure, and visual organization 2. Analyze the textual content: key phrases, terminology, purpose, and information type 3. Identify distinctive features that match the document type descriptions 4. Decide if this page starts a new document (output \"start\") or continues the previous document (output \"continue\") 5. Consider both visual and textual evidence together to determine the best match 6. CRITICAL: Only use document types explicitly listed in the <document-types> section </classification-instructions>\n<output-format> {\n  \"classification_reason\": \"Detailed reasoning including specific visual and textual evidence that led to this classification\",\n  \"class\": \"exact_document_type_from_list\",\n  \"document_boundary\": \"start or continue\"\n} </output-format>\n<<CACHEPOINT>>\n<document-ocr-data> {DOCUMENT_TEXT} </document-ocr-data>\n<document-image> {DOCUMENT_IMAGE} </document-image>\n<final-instructions> Analyze the document above by: 1. Applying the <classification-instructions> to examine both visual and textual features 2. Selecting ONLY from document types in <document-types> 3. Providing clear reasoning with specific evidence 4. Outputting in the exact JSON format specified in <output-format> </final-instructions>",
        "temperature": "0.0",
        "top_k": "5",
        "top_p": "0.0"
    },
    "extraction": {
        "confidence": {
            "mode": "integrated"
        },
        "agentic": {
            "enabled": False,
            "review_agent": False
        },
        "image": {
            "target_height": 1200,
            "target_width": 1000
        },
        "max_tokens": "40000",
        "model": "us.anthropic.claude-opus-4-6-v1",
        "system_prompt": "You are a document assistant. Respond only with JSON. Never make up data, only provide data found in the document being provided.",
        "task_prompt": "<background> You are an expert in document analysis and information extraction. You can understand and extract key information from documents classified as type: \n{DOCUMENT_CLASS}. </background>\n<task> Provide your 4 best guesses and the probability that each is correct (0.0 to 1.0) for each attribute listed below. Give ONLY the guesses and probabilities, no other words or explanation.</task> \n<extraction-guidelines> Guidelines:\n 1. All dates should be in MM/DD/YY format\n 2. Only consider values that are present in the provided document. Do not invent attribute values.3. Exract each entity candidate exactly how it is presented in the document. E.g. do not change \"M\" to \"Monday\" or change the date from \"01/04\" to \"1/04\".4. For each extracted attribute key, the value MUST be a nested object formatted exactly like this:\n        {\n          \"G1\": \"<1st most likely guess, as short as possible>\",\n          \"P1\": <probability between 0.0 and 1.0>,\n          \"G2\": \"<2nd most likely guess, as short as possible>\",\n          \"P2\": <probability between 0.0 and 1.0>,\n          \"G3\": \"<3rd most likely guess, as short as possible>\",\n          \"P3\": <probability between 0.0 and 1.0>,\n          \"G4\": \"<4th most likely guess, as short as possible>\",\n          \"P4\": <probability between 0.0 and 1.0>\n        }\n</extraction-guidelines>\nIf the attributes section below contains a list of attribute names and descriptions, then output only those attributes, using the provided descriptions as guidance for finding the correct values. \n<attributes> {ATTRIBUTE_NAMES_AND_DESCRIPTIONS} </attributes>\n<<CACHEPOINT>>\n<document_image> {DOCUMENT_IMAGE} </document_image>\n<output-format>\n<ocr-text-confidence-results> {OCR_TEXT_CONFIDENCE} </ocr-text-confidence-results>\nGive ONLY the JSON object containing the guesses and probabilities. Do not think step-by-step, do not show your work, and do not provide any reasoning, justifications, or extra commentary whatsoever.\nFor SIMPLE attributes: { \"simple_attribute_name\": { \"G1\": \"<1st most likely guess, as short as possible>\", \"P1\": <probability between 0.0 and 1.0>, \"G2\": \"<2nd most likely guess, as short as possible>\", \"P2\": <probability between 0.0 and 1.0>, \"G3\": \"<3rd most likely guess, as short as possible>\", \"P3\": <probability between 0.0 and 1.0>, \"G4\": \"<4th most likely guess, as short as possible>\", \"P4\": <probability between 0.0 and 1.0> } }\nFor LIST attributes (assess EACH item individually): { \"list_attribute_name\": [ { \"item_attribute_1\": { \"G1\": \"<1st guess>\", \"P1\": <probability>, \"G2\": \"<2nd guess>\", \"P2\": <probability>, \"G3\": \"<3rd guess>\", \"P3\": <probability>, \"G4\": \"<4th guess>\", \"P4\": <probability> }, \"item_attribute_2\": { \"G1\": \"<1st guess>\", \"P1\": <probability>, \"G2\": \"<2nd guess>\", \"P2\": <probability>, \"G3\": \"<3rd guess>\", \"P3\": <probability>, \"G4\": \"<4th guess>\", \"P4\": <probability> } } ] } </output-format>",
        "temperature": "0.0",
        "top_k": "5",
        "top_p": "0.0"
    },
    "assessment": {
        "enabled": False
    },
    "classes": [
        {
            "$schema": "https://json-schema.org/draft/2020-12/schema",
            "$defs": {
                "LineItem": {
                    "type": "object",
                    "properties": {
                        "LineItemStartDate": {
                            "description": "The date marking the beginning of the scheduled airing period for a line item. May appear as a standalone column (e.g., \"Start Date\"), as the first part of a date range (e.g., \"10/13/14 to 10/19/14\"), or embedded within a \"Schedule Days to Run\" or \"Airtime\" or \"Scheduled\" field. Formats include MM/DD/YY, MM/DD/YYYY etc. In some documents, this appears at the line-item level; in others, it appears at a sub-line/week level beneath the main line entry. Output null if not shown.",
                            "x-aws-idp-confidence-threshold": "0.8",
                            "x-aws-idp-evaluation-method": "LEVENSHTEIN",
                            "x-aws-idp-evaluation-threshold": "0.7",
                            "type": "string"
                        },
                        "LineItemDays": {
                            "description": "The days of the week on which the spot is scheduled to air. This field has highly variable formatting. Example 1, dash-letter pattern (7-character positional):Each position represents M-T-W-T-F-S-S, with a letter indicating the day is active and a dash indicating it's not. Example 1: \"-TWTF--\" (Tue to Fri), \"1------\" (Mon only), \"------1\" (Sun only), \"-----S-\" (Sat), \"----1--\" (Fri). Example 2: Abbreviated day names on sublines:Individual spot lines may list days as \"Tu\", \"W\", \"Th\", \"Mon\", \"Tue\", etc. Example 3: Comma-separated compact codes such as \"Day,Th-1\", \"Day,F-2\". Example 4: It may also appear as descriptive prefixes in the description field, e.g., \"M-F\" indicating Monday through Friday. You MUST extract exactly as given in the document.",
                            "x-aws-idp-confidence-threshold": "0.8",
                            "x-aws-idp-evaluation-method": "LEVENSHTEIN",
                            "x-aws-idp-evaluation-threshold": "0.7",
                            "type": "string"
                        },
                        "LineItemEndDate": {
                            "description": "The date marking the end of the scheduled airing period for a line item. Appears as a standalone column (\"End Date\"), or as the second part of a date range (e.g., \"10/13/14 to 10/19/14\", \"2/1/2016 - 2/28/2016\"). Formats mirror those of LineItemStartDate. The end date may be identical to the start date when a line item covers a single day. Output null if not explicitly stated.",
                            "x-aws-idp-confidence-threshold": "0.8",
                            "x-aws-idp-evaluation-method": "LEVENSHTEIN",
                            "x-aws-idp-evaluation-threshold": "0.7",
                            "type": "string"
                        },
                        "LineItemDescription": {
                            "description": "The name or title of the program, show, or time slot in which the advertisement is scheduled to air. Examples include program names (\"People's Court\", \"Dr. Phil\", \"Rachael Ray\", \"NFL on FOX 3p\"), newscast identifiers (\"M-F 10P-1035P NEWS\", \"SA 6P-630P NEWS\", \"SU 10P-1035P NEWS\"), time-slot descriptions (\"M-F 1135p-1205a\", \"Su 1205p-1235a\"), or simulcast/edition names (\"Noticias 45 Ed Nocturna\", \"C-News Simulcast\"). May include channel prefixes, day prefixes (e.g., \"SA\", \"SU\", \"M-F\"). Can also contain supplementary qualifiers like \"65572-[LCL]\". Output null if not shown.",
                            "x-aws-idp-evaluation-method": "LEVENSHTEIN",
                            "x-aws-idp-evaluation-threshold": "0.7",
                            "type": "string"
                        },
                        "LineItemRate": {
                            "description": "The rate in dollars per show in the line entry. Only extract the number. E.g. if the rate is \"$800.15\", extract 800.15. Output null if not shown.",
                            "x-aws-idp-confidence-threshold": "0.8",
                            "x-aws-idp-evaluation-method": "NUMERIC_EXACT",
                            "type": "number"
                        }
                    }
                }
            },
            "description": "Invoice document",
            "type": "object",
            "x-aws-idp-document-type": "Invoice",
            "properties": {
                "Agency": {
                    "description": "The agency the invoice is addressed to",
                    "x-aws-idp-confidence-threshold": "0.8",
                    "x-aws-idp-evaluation-weight": "2",
                    "x-aws-idp-evaluation-method": "LEVENSHTEIN",
                    "x-aws-idp-evaluation-threshold": "0.7",
                    "type": "string"
                },
                "Advertiser": {
                    "description": "The name of the advertiser",
                    "x-aws-idp-confidence-threshold": "0.8",
                    "x-aws-idp-evaluation-weight": "2",
                    "x-aws-idp-evaluation-method": "FUZZY",
                    "x-aws-idp-evaluation-threshold": "0.8",
                    "type": "string"
                },
                "GrossTotal": {
                    "description": "The gross total amount. Output null if not shown.",
                    "x-aws-idp-evaluation-weight": "2",
                    "x-aws-idp-confidence-threshold": "0.8",
                    "x-aws-idp-evaluation-method": "NUMERIC_EXACT",
                    "type": "number"
                },
                "PaymentTerms": {
                    "description": "Terms of payment. Output null if not shown.",
                    "x-aws-idp-evaluation-weight": "0.2",
                    "x-aws-idp-evaluation-method": "FUZZY",
                    "x-aws-idp-evaluation-threshold": "0.7",
                    "type": "string"
                },
                "AgencyCommission": {
                    "description": "The agency commission amount. Output null if not shown.",
                    "x-aws-idp-evaluation-weight": "0.2",
                    "x-aws-idp-confidence-threshold": "0.8",
                    "x-aws-idp-evaluation-method": "NUMERIC_EXACT",
                    "type": "number"
                },
                "NetAmountDue": {
                    "description": "The net amount due after commission. Output null if not shown.",
                    "x-aws-idp-evaluation-weight": "2",
                    "x-aws-idp-confidence-threshold": "0.8",
                    "x-aws-idp-evaluation-method": "NUMERIC_EXACT",
                    "type": "number"
                },
                "LineItems": {
                    "type": "array",
                    "description": "List of line item details on the invoice; each item has several possible elements",
                    "items": {
                        "$ref": "#/$defs/LineItem"
                    }
                }
            },
            "required": [
                "Agency",
                "Advertiser",
                "LineItems"
            ],
            "$id": "Invoice"
        }
    ]
}

print("1S-TopK Configuration loaded")
print(f"  Extraction model: {CONFIG['extraction']['model']}")
print(f"  Classification model: {CONFIG['classification']['model']}")
print(f"  Document classes: {[c.get('x-aws-idp-document-type', c.get(chr(36)+chr(105)+chr(100))) for c in CONFIG['classes']]}")
print(f"  Assessment separate step: DISABLED (handled by extraction)")


1S-TopK Configuration loaded
  Extraction model: us.anthropic.claude-opus-4-6-v1
  Classification model: us.amazon.nova-pro-v1:0
  Document classes: ['Invoice']
  Assessment separate step: DISABLED (handled by extraction)


## 5. Process Document with OCR

In [5]:
# Initialize a new Document
document = Document(
    id="fcc-invoice-1s-topk",
    input_bucket=input_bucket_name,
    input_key=sample_file_key,
    output_bucket=output_bucket_name,
    status=Status.QUEUED
)

print(f"Created document with ID: {document.id}")
print(f"Status: {document.status.value}")

# Create OCR service with Textract
ocr_service = ocr.OcrService(
    region=region,
    enhanced_features=['LAYOUT']
)

# Process document with OCR
print("\nProcessing document with OCR...")
start_time = time.time()
document = ocr_service.process_document(document)
ocr_time = time.time() - start_time

print(f"OCR processing completed in {ocr_time:.2f} seconds")
print(f"Document status: {document.status.value}")
print(f"Number of pages processed: {document.num_pages}")

# Show pages information
print("\nProcessed pages:")
for page_id, page in document.pages.items():
    print(f"Page {page_id}: Image URI: {page.image_uri}")
print("\nMetering:")
print(json.dumps(document.metering))

INFO:idp_common.ocr.service:OCR Service initialized - DPI: None, No image sizing limits
INFO:idp_common.ocr.service:OCR Service initialized with features: ['LAYOUT']
INFO:idp_common.ocr.service:OCR Service initialized with Textract backend
INFO:idp_common.ocr.service:S3 client initialized with 20 connection pool size


Created document with ID: fcc-invoice-1s-topk
Status: QUEUED

Processing document with OCR...


INFO:idp_common.ocr.service:Detected file type: pdf
INFO:idp_common.ocr.service:Rendering 6 of 6 page images sequentially (pypdfium2 is not thread-safe)
INFO:idp_common.ocr.service:Page 1 extracted at original size: 1275x1651
INFO:idp_common.ocr.service:Page 2 extracted at original size: 1275x1651
INFO:idp_common.ocr.service:Page 3 extracted at original size: 1275x1651
INFO:idp_common.ocr.service:Page 4 extracted at original size: 1275x1651
INFO:idp_common.ocr.service:Page 5 extracted at original size: 1275x1651
INFO:idp_common.ocr.service:Page 6 extracted at original size: 1275x1651
INFO:idp_common.ocr.service:Rendered 6 page images
INFO:idp_common.ocr.service:Memory usage: 610.2 MB
INFO:idp_common.ocr.service:Successfully extracted markdown text for page 3
INFO:idp_common.ocr.service:Successfully extracted markdown text for page 1
INFO:idp_common.ocr.service:Successfully extracted markdown text for page 6
INFO:idp_common.ocr.service:Successfully extracted markdown text for page 5
INF

OCR processing completed in 7.96 seconds
Document status: QUEUED
Number of pages processed: 6

Processed pages:
Page 1: Image URI: s3://idp-notebook-assess-output-195275636621-us-west-2/sample-assessment-2026-07-07_21-39-43.pdf/pages/1/image.jpg
Page 2: Image URI: s3://idp-notebook-assess-output-195275636621-us-west-2/sample-assessment-2026-07-07_21-39-43.pdf/pages/2/image.jpg
Page 3: Image URI: s3://idp-notebook-assess-output-195275636621-us-west-2/sample-assessment-2026-07-07_21-39-43.pdf/pages/3/image.jpg
Page 4: Image URI: s3://idp-notebook-assess-output-195275636621-us-west-2/sample-assessment-2026-07-07_21-39-43.pdf/pages/4/image.jpg
Page 5: Image URI: s3://idp-notebook-assess-output-195275636621-us-west-2/sample-assessment-2026-07-07_21-39-43.pdf/pages/5/image.jpg
Page 6: Image URI: s3://idp-notebook-assess-output-195275636621-us-west-2/sample-assessment-2026-07-07_21-39-43.pdf/pages/6/image.jpg

Metering:
{"OCR/textract/analyze_document-Layout": {"pages": 6}}


## 6. Classify the Document

In [6]:
# Create classification service with Bedrock backend
classification_service = classification.ClassificationService(
    config=CONFIG, 
    backend="bedrock" 
)

# Classify the document
print("\nClassifying document...")
start_time = time.time()
document = classification_service.classify_document(document)
classification_time = time.time() - start_time
print(f"Classification completed in {classification_time:.2f} seconds")
print(f"Document status: {document.status.value}")

# Show classification results
if document.sections:
    print("\nDetected sections:")
    for section in document.sections:
        print(f"Section {section.section_id}: {section.classification}")
        print(f"  Pages: {section.page_ids}")
else:
    print("\nNo sections detected")

# Show page classification
print("\nPage-level classifications:")
for page_id, page in sorted(document.pages.items()):
    print(f"Page {page_id}: {page.classification}")


Classifying document...
Classification completed in 0.00 seconds
Document status: QUEUED

Detected sections:
Section 1: Invoice
  Pages: ['1', '2', '3', '4', '5', '6']

Page-level classifications:
Page 1: Invoice
Page 2: Invoice
Page 3: Invoice
Page 4: Invoice
Page 5: Invoice
Page 6: Invoice


## 7. Extract Information & Get Corresponding Confidence Measures

In [7]:
# Create extraction service with Bedrock
extraction_service = extraction.ExtractionService(config=CONFIG)

print("\nExtracting information from document sections...")

n = 3 # Only process first 3 sections to save time
# Process each section directly using the section_id
for section in document.sections[:n]:  
    print(f"\nProcessing section {section.section_id} (class: {section.classification})")
    
    # Process section directly with the original document
    start_time = time.time()
    document = extraction_service.process_document_section(
        document=document,
        section_id=section.section_id
    )
    
    extraction_time = time.time() - start_time
    print(f"Extraction for section {section.section_id} completed in {extraction_time:.2f} seconds")
    
print(f"\nExtraction for first {n} sections complete.")




Extracting information from document sections...

Processing section 1 (class: Invoice)


DEBUG:idp_common.bedrock.client:Found <<CACHEPOINT>> tags in text content: <background> You are an expert in document analysi...
DEBUG:idp_common.bedrock.client:Split text into 2 parts at cachepoint tags
DEBUG:idp_common.bedrock.client:Text part 1: 759 words
DEBUG:idp_common.bedrock.client:Inserting cachePoint #1 after text part 1
DEBUG:idp_common.bedrock.client:Text part 2: 1 words
DEBUG:idp_common.bedrock.client:No cachepoint tags in image content, passing through unchanged
DEBUG:idp_common.bedrock.client:No cachepoint tags in image content, passing through unchanged
DEBUG:idp_common.bedrock.client:No cachepoint tags in image content, passing through unchanged
DEBUG:idp_common.bedrock.client:No cachepoint tags in image content, passing through unchanged
DEBUG:idp_common.bedrock.client:No cachepoint tags in image content, passing through unchanged
DEBUG:idp_common.bedrock.client:No cachepoint tags in image content, passing through unchanged
DEBUG:idp_common.bedrock.client:No cachepoin

Extraction for section 1 completed in 67.36 seconds

Extraction for first 3 sections complete.


In [8]:
# Load the updated extraction results with assessment
extraction_data = load_json_from_s3(section.extraction_result_uri)
        
# Display the inference results
print(f"  Extraction Results:")
inference_result = extraction_data.get('inference_result', {})

for attr_name, attr_value in inference_result.items():
    if isinstance(attr_value, list):
        print(f"    {attr_name}: [{len(attr_value)} items]")
        for i, item in enumerate(attr_value[:3]):
            print(f"      [{i}] {item}")
    else:
        print(f"    {attr_name}: {attr_value}")

# Print the original structured output with top 4 candidates and corresponding probabilities
print("\n\nTopK candidate guesses G_i and their probabilities P_i\n")
topk_candidates = extraction_data.get("metadata", {}).get("topk_candidates", {})
for attr_name, attr_value in topk_candidates.items():
    if isinstance(attr_value, list):
        print(f"    {attr_name}: [{len(attr_value)} items]")
        for i, item in enumerate(attr_value[:3]):
            print(f"      [{i}] {item}")
    else:
        print(f"    {attr_name}: {attr_value}")

  Extraction Results:
    Agency: NCC
    Advertiser: JASON ALLEN FOR CONGRESS MI CD 1
    GrossTotal: 2319.0
    PaymentTerms: CASH IN ADVANCE
    AgencyCommission: 347.85
    NetAmountDue: 1714.9
    LineItems: [14 items]
      [0] {'LineItemStartDate': '08/01/16', 'LineItemDays': None, 'LineItemEndDate': '08/02/16', 'LineItemDescription': 'AEN', 'LineItemRate': 60.0}
      [1] {'LineItemStartDate': '08/01/16', 'LineItemDays': None, 'LineItemEndDate': '08/02/16', 'LineItemDescription': 'AMC', 'LineItemRate': 49.0}
      [2] {'LineItemStartDate': '08/01/16', 'LineItemDays': None, 'LineItemEndDate': '08/02/16', 'LineItemDescription': 'APL', 'LineItemRate': 22.0}


TopK candidate guesses G_i and their probabilities P_i

    Agency: {'G1': 'NCC', 'P1': 0.95, 'G2': 'NCC R18', 'P2': 0.03, 'G3': 'R18', 'P3': 0.01, 'G4': 'SPECTRUM REACH', 'P4': 0.01}
    Advertiser: {'G1': 'JASON ALLEN FOR CONGRESS MI CD 1', 'P1': 0.85, 'G2': 'JASON ALLEN FOR CONGRESS', 'P2': 0.12, 'G3': 'CIA86882', 'P3': 0.

## 9. Display Assessment Results

Let's examine the assessment results that have been added to the extraction results.

In [9]:
print("Assessment Results:")
print("===================")

for section in document.sections[:n]:
    if section.extraction_result_uri:
        print(f"Section {section.section_id} ({section.classification}):")
        
        # Load the updated extraction results with assessment
        extraction_data = load_json_from_s3(section.extraction_result_uri)
        
        # Display the inference results
        print(f"  Extraction Results:")
        inference_result = extraction_data.get('inference_result', {})
        for attr_name, attr_value in inference_result.items():
            if isinstance(attr_value, list):
                print(f"    {attr_name}: [{len(attr_value)} items]")
                for i, item in enumerate(attr_value[:3]):
                    print(f"      [{i}] {item}")
            else:
                print(f"    {attr_name}: {attr_value}")
        
        # Display the assessment results (explainability_info)
        explainability_info = extraction_data.get('explainability_info', [])
        if explainability_info:
            print(f"  Confidence Assessment (1S-TopK):")
            for attr_name, assessment in explainability_info[0].items():
                if isinstance(assessment, dict) and 'confidence' in assessment:
                    conf = assessment['confidence']
                    threshold = assessment.get('confidence_threshold', 'N/A')
                    status = "✓" if conf >= float(threshold) else "⚠ BELOW THRESHOLD"
                    print(f"    {attr_name}: {conf:.2f} (threshold: {threshold}) {status}")
                elif isinstance(assessment, list):
                    # List attribute (e.g., LineItems) - per-item assessments
                    print(f"    {attr_name}: [{len(assessment)} items]")
                    for i, item_assess in enumerate(assessment[:3]):
                        if isinstance(item_assess, dict):
                            for sub_attr, sub_val in item_assess.items():
                                if isinstance(sub_val, dict) and 'confidence' in sub_val:
                                    conf = sub_val['confidence']
                                    threshold = sub_val.get('confidence_threshold', 'N/A')
                                    print(f"      [{i}].{sub_attr}: {conf:.2f} (threshold: {threshold})")
                else:
                    print(f"    {attr_name}: {assessment}")
        else:
            print(f"  No assessment results found")
        
        print()


Assessment Results:
Section 1 (Invoice):
  Extraction Results:
    Agency: NCC
    Advertiser: JASON ALLEN FOR CONGRESS MI CD 1
    GrossTotal: 2319.0
    PaymentTerms: CASH IN ADVANCE
    AgencyCommission: 347.85
    NetAmountDue: 1714.9
    LineItems: [14 items]
      [0] {'LineItemStartDate': '08/01/16', 'LineItemDays': None, 'LineItemEndDate': '08/02/16', 'LineItemDescription': 'AEN', 'LineItemRate': 60.0}
      [1] {'LineItemStartDate': '08/01/16', 'LineItemDays': None, 'LineItemEndDate': '08/02/16', 'LineItemDescription': 'AMC', 'LineItemRate': 49.0}
      [2] {'LineItemStartDate': '08/01/16', 'LineItemDays': None, 'LineItemEndDate': '08/02/16', 'LineItemDescription': 'APL', 'LineItemRate': 22.0}
  Confidence Assessment (1S-TopK):
    Agency: 0.95 (threshold: 0.8) ✓
    Advertiser: 0.85 (threshold: 0.8) ✓
    GrossTotal: 0.92 (threshold: 0.8) ✓
    PaymentTerms: 0.95 (threshold: 0.8) ✓
    AgencyCommission: 0.85 (threshold: 0.8) ✓
    NetAmountDue: 0.92 (threshold: 0.8) ✓
    Lin

## 10. Final Document Status Summary

Summary of the 1S-TopK pipeline run.


In [10]:
# Update document status to COMPLETED
document.status = Status.COMPLETED

# Display final document state
print("Final Document State:")
print(f"Document ID: {document.id}")
print(f"Status: {document.status.value}")
print(f"Number of pages: {document.num_pages}")
print(f"Number of sections: {len(document.sections)}")

print("=== 1S-TopK Pipeline Summary ===")
print("✅ OCR Processing - Convert PDF to text and images")
print("✅ Document Classification - Identify document types")
print("✅ 1S-TopK Extraction + Confidence elicitation - Single LLM call produces both:")
print("   • Extracted values (top guess G1 per attribute)")
print("   • Confidence scores (probability P1 per attribute, 0.0–1.0)")
print("   • Alternative candidates (G2–G4 with P2–P4)")
print("⏭️  Separate Assessment - SKIPPED (already handled by extraction)")


Final Document State:
Document ID: fcc-invoice-1s-topk
Status: COMPLETED
Number of pages: 6
Number of sections: 1
=== 1S-TopK Pipeline Summary ===
✅ OCR Processing - Convert PDF to text and images
✅ Document Classification - Identify document types
✅ 1S-TopK Extraction + Confidence elicitation - Single LLM call produces both:
   • Extracted values (top guess G1 per attribute)
   • Confidence scores (probability P1 per attribute, 0.0–1.0)
   • Alternative candidates (G2–G4 with P2–P4)
⏭️  Separate Assessment - SKIPPED (already handled by extraction)


## Conclusion

This notebook demonstrates the **1S-TopK** approach for combined extraction + confidence elicitation in a single LLM call.

### Why Top-K Candidates?

Asking the LLM for **multiple ranked guesses with probabilities** produces better-calibrated confidence than simply asking "how confident are you?":

- **Distributes probability mass** — The model must weigh alternatives rather than defaulting to near-1.0 confidence on uncertain extractions
- **Verbalized confidence works** — Tian et al. (EMNLP 2023) showed that verbalized probabilities from RLHF-LMs are ~50% better-calibrated than internal token probabilities of those RHLF-LMs on Q-A datasets that they experimented on.

### 1S vs 2S: Trade-offs

| | 1-Step (1S-TopK) | 2-Step (separate assessment) |
|---|---|---|
| LLM calls | 1 per section | 2 per section |
| Latency | Lower | Higher |
| Cost | Lower (single inference) | Higher (two inferences) |
| Task complexity per call | Higher (extraction + confidence together) | Lower (each call has a focused task) |

### Caveats

For **complex or long documents**, combining extraction and confidence elicitation in a single call increases the task complexity for the LLM. This may impact the quality of either the extraction or the confidence scores (or both) compared to a two-step approach where each call has a narrower, focused objective.

We are **not claiming that the 1S method always outperforms 2S**. This is an alternative approach. Choose the method that best suits your dataset and quality requirements.
